In this experiment we investigate the tradeoff using the gating mechanism introduced. We precompute the results of not searching and applying the search for faster evalution of the thresholds. We then vary the thresholds and test how the accuracy of a transformed version of the test set behaves and an untransformed version.

We do this due to the lack of having a realistic transformed versions where only some of the data has transformtions. In a realistic scenario one would select the threshold on the validation set first. We illustrate this in the end where we generate mixtures of the dataset and fit the threshold on there(using mean accuracy over multiple runs). Depending on the mixture and dataset this can improve the accuracy a lot while also only requiring the search on a subset of the data.

We generally inverstige the best ood detector from the previos experiment, a learned energy model and the case where one trains a second model using augmentation.
If refitting is possible augmentation is preferable on the tested combinations.
If not one can still have good gains in accuracy using canonicalization.

In the end we als provide a short check method that does not use the precomputation but is given the found threshold.

In [ ]:
#Important

In [ ]:
import torch
from src.utils.sampling import BatchNegativeSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "bigger_emnist"

default_architecutre_mapping = {
    "mnist": "resnet_small",
    "bigger_mnist": "resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist": "bigger_extended_resnet_small",
    "coil100": "coil_resnet_small",
    "tu_berlin": "bi_lstm",
    "modelnet10": "pointnetplus",
}

architecture = default_architecutre_mapping[dataset]
budget = None

In [ ]:
import torch
from numpy.testing import assert_array_equal


def test_loader_determinism(loader, num_batches_to_test: int = 5):
    """Tests if iterating a PyTorch DataLoader twice yields identical results."""

    print(f"Testing loader determinism over the first {num_batches_to_test} batches...")

    run1_data, run1_labels = [], []
    loader_iter1 = iter(loader)
    for _ in tqdm.tqdm(range(num_batches_to_test), desc="Run 1"):
        try:
            x, y = next(loader_iter1)
            run1_data.append(x)
            run1_labels.append(y)
        except StopIteration:
            break

    run2_data, run2_labels = [], []
    loader_iter2 = iter(loader)
    for _ in tqdm.tqdm(range(num_batches_to_test), desc="Run 2"):
        try:
            x, y = next(loader_iter2)
            run2_data.append(x)
            run2_labels.append(y)
        except StopIteration:
            break

    if len(run1_data) != len(run2_data):
        print(f"Error: Run 1 yielded {len(run1_data)} batches, Run 2 yielded {len(run2_data)} batches.")
        return False

    for i in range(len(run1_data)):
        try:
            assert_array_equal(run1_labels[i].numpy(), run2_labels[i].numpy())
        except AssertionError as e:
            print(f"Different values in Batch {i}: Labels are not identical.")
            print(e)
            return False

        try:
            assert torch.allclose(run1_data[i], run2_data[i], atol=1e-6)
        except AssertionError as e:
            print(
                f"Different values in Batch {i}: Data tensors are not identical (max diff: {(run1_data[i] - run2_data[i]).abs().max()}.")
            print("This indicates randomness in the data loading or augmentation pipeline.")
            return False
    return True



In [ ]:
from data.get_dataset import get_dataset_info, get_dataset

dataset_info = get_dataset_info(dataset)

dataset_dict = get_dataset(dataset_info, path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:


dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

In [ ]:
x = next(iter(test_loader_transformed))[0]

batch_size = next(iter(train_loader))[0].shape[0]

from src.utils.eval.vis import vis_dataset, plt_setup_paper

vis_dataset(train_loader, val_loader, test_loader_transformed)
from model.train import train_and_get_model, train_or_load_energy_model
from model.get_model import get_network
from src.utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,
                                "comparison_unsupervised")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, transform_name, f"{safe}.json")


def cache_path(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(embedding_cache_path, dataset, architecture, transform_name, f"{safe}_embedding_cache.npz")

In [ ]:
model = get_network(dataset_info, architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train = f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model, model_dir_path, modelname, train_loader, val_loader, trainer_kwargs={
    "accelerator": "auto",
    "max_epochs": dataset_info.epochs,
    "precision": "16-mixed",
}, load_if_exists=True)



In [ ]:
model.eval().to(device)

In [ ]:
#check main model
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)
res = evaluate_base_model(model, test_loader, device)
print(res)

In [ ]:
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]

from src.utils.replacer import replace_rotation_transforms_2vec

from data.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)

if dataset == "modelnet10":
    transform_seq = replace_rotation_transforms_2vec(transform_seq)

In [ ]:
print(transform_seq.transformations)

In [ ]:

from model.basic_networks import make_deterministic

make_deterministic(model)



In [ ]:
from get_dataset import get_layer_embedding_cache_config, create_layer_embedding_cache

cache_config = get_layer_embedding_cache_config(dataset, architecture, transform_name=None, dataset_info=dataset_info)
train_cache = create_layer_embedding_cache(model, train_loader_no_shuffle, cache_config, embedding_cache_path,
                                           device=device)


In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
from model.basic_networks import FlexibleResNet

detectors = ["knn", "per_class_knn", "knn_mixed", "per_class_knn_mixed", "knn_mixed_faiss", "knn_itf", "vim", "react",
             "dice", "ash", "she", "laplace_mi", "laplace_energy", "laplace_weighted", "trust_score", "openmax",
             "mahalanobis", "rmd", "class_prototype", "react_all", "energy", "per_class_prototype",
             "single_mahalanobis", "single_rmd", "single_mahalanobis_individual", "single_rmd_individual",
             "mahalanobis_individual", "rmd_individual", "prototype_multi", "laplace_entropy", "adjusted_entropy",
             "nn_guided", "nn_guided_one"]

detectors_parameterless = ["energy_ts", "entropy", "kl_matching", "laplace_entropy_gridsearch"]
detectors = detectors + detectors_parameterless
if not isinstance(model, FlexibleResNet):
    #remove ash and react_all as they only work with resnets in our implementation
    detectors.remove("ash")
    detectors.remove("react_all")
    if "react" not in detectors:
        detectors.append("react")
    detectors.append("ash_last")



In [ ]:

import os
import math
from typing import Tuple, Optional, Any, Dict

from hyper_param.ood.base_prepare import create_ood_problem, get_default_ood_params, find_best_detector_and_instantiate

base_results_dir = os.path.join(
    current_path,
    "experiment_files",
    "results",
    "comparison_unsupervised",
    str(dataset),
    str(architecture),
    getattr(dataset_info, "transform_seq_name", "default"),
)

best_detector, best_problem, best_score, second_choice, second_problem, second_score = find_best_detector_and_instantiate(
    base_results_dir=base_results_dir,
    detectors=detectors,
    model=model,
    train_cache=train_cache,
    transform_seq_arg=transform_seq,
    dataset_info=dataset_info,
    architecture=architecture,
    device=device,
    val_id_loader=val_loader_transformed,
    val_ood_loader=val_loader_transformed,
)

In [ ]:
#store the best detector for each dataset to ensure we use the same for all experiments



In [ ]:
#here we do the following calculate the accuracy on the test set, then calculate on the transformed test set. We also store the confidence valeus
#then we do the optimiztation to find which ones we can transform to get better accuracy.
#we then draw the curve over the threshold when we switch from doing no optimization to doing optimization, this should drop the accuracy on the test set but increase on the transformed test set.



In [ ]:
conf = best_problem.confidence_module

In [ ]:
import os, json

results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,
                                "comparison_augmented_models")
result_path = os.path.join(results_dir_path, transform_name, f"{modelname}_augmented.json")


#if dataset =="modelnet10":
#    result_path = os.path.join(results_dir_path, transform_name, #f"modelnet10_pointnetplus_pointnetplus_euclidean_pca.json")
#if dataset == "tu_berlin":
#    result_path = os.path.join(results_dir_path, transform_name, f"{modelname}_pca.json")


with open(result_path, "r") as f: results = json.load(f)
print(results)

augmnet_acc_on_in_dist = results.get("original_test_results", None)["accuracy"]

In [ ]:
import os
import numpy as np
import torch
import tqdm
import matplotlib.pyplot as plt
from typing import Dict


@torch.no_grad()
def calculate_confidences_and_cache(
        optimizer,
        best_problem,
        conf_fn,
        test_loader,
        test_loader_transformed,
        device,
        n_runs: int = 5,
        cache_path: str = "cached_confidences.npz",
        force_recompute: bool = False,
        augment_true_func=None,
) -> Dict[str, np.ndarray]:
    """
    Calculates and caches:
      - base confidence/correctness (without optimization)
      - optimized confidence/correctness (repeated n_runs times for stochastic optimizer)
    If `cache_path` exists, loads results unless `force_recompute=True`.
    """
    test_loader_determinism(test_loader)
    test_loader_determinism(test_loader_transformed)

    if os.path.exists(cache_path) and not force_recompute:
        print(f" Loading cached results from {cache_path}")
        return dict(np.load(cache_path, allow_pickle=True))

    print("=== Calculating base confidences (no optimization) ===")

    def eval_conf(loader, augment_func=None):
        all_conf, correct = [], []
        for x_test, y_test in tqdm.tqdm(loader):
            x_test = x_test.to(device)
            if augment_func is not None:
                x_test = augment_func(x_test)
            conf_values, logits = conf_fn(x_test)
            if logits is None:
                logits = model(x_test)
            pred_y = torch.argmax(logits, dim=1).cpu()
            correct.append((pred_y == y_test).numpy())
            all_conf.append(conf_values.cpu().numpy())
        return np.concatenate(correct), np.concatenate(all_conf)

    correct_predicted, all_conf = eval_conf(test_loader, augment_true_func)
    correct_predicted_transformed, all_conf_transformed = eval_conf(test_loader_transformed)

    print(all_conf.min())
    print(all_conf.min())

    results = {
        "correct_predicted": correct_predicted,
        "all_conf": all_conf,
        "correct_predicted_transformed": correct_predicted_transformed,
        "all_conf_transformed": all_conf_transformed,
    }

    print("\n=== Running probabilistic optimizer ===")

    optimized_correct_runs, optimized_conf_runs = [], []
    optimized_correct_runs_t, optimized_conf_runs_t = [], []

    for run_idx in range(n_runs):
        print(f"\n--- Optimization run {run_idx + 1}/{n_runs} ---")

        correct_run, conf_run = [], []
        for x_test, y_test in tqdm.tqdm(test_loader, desc=f"Test run {run_idx + 1}"):
            x_test = x_test.to(device)
            p, error, _ = optimizer.optimize(best_problem, x_test)
            x_optimized = best_problem.transform_sequence.transform(x_test, p)
            conf_values, logits = conf_fn(x_optimized)
            if logits is None:
                logits = model(x_optimized)
            pred_y = torch.argmax(logits, dim=1).cpu()
            correct_run.append((pred_y == y_test).numpy())
            conf_run.append(conf_values.cpu().numpy())
        optimized_correct_runs.append(np.concatenate(correct_run))
        optimized_conf_runs.append(np.concatenate(conf_run))

        correct_run_t, conf_run_t = [], []
        for x_test, y_test in tqdm.tqdm(test_loader_transformed, desc=f"Transformed run {run_idx + 1}"):
            x_test = x_test.to(device)
            p, error, _ = optimizer.optimize(best_problem, x_test)
            x_optimized = best_problem.transform_sequence.transform(x_test, p)
            conf_values, logits = conf_fn(x_optimized)
            if logits is None:
                logits = model(x_optimized)
            pred_y = torch.argmax(logits, dim=1).cpu()
            correct_run_t.append((pred_y == y_test).numpy())
            conf_run_t.append(conf_values.cpu().numpy())
        optimized_correct_runs_t.append(np.concatenate(correct_run_t))
        optimized_conf_runs_t.append(np.concatenate(conf_run_t))

    results["optimized_correct_predicted"] = np.stack(optimized_correct_runs, axis=1)
    results["optimized_all_conf"] = np.stack(optimized_conf_runs, axis=1)
    results["optimized_correct_predicted_transformed"] = np.stack(optimized_correct_runs_t, axis=1)
    results["optimized_all_conf_transformed"] = np.stack(optimized_conf_runs_t, axis=1)

    np.savez_compressed(cache_path, **results)
    print(f"\nCached results saved to: {cache_path}")

    return results


import matplotlib.ticker as mticker
import numpy as np
import matplotlib.pyplot as plt

W = plt_setup_paper()


def plot_confidence_threshold_results(
        results: dict,
        dataset_name: str = "Dataset",
        use_only_improved: bool = True,
        improvement_metric: str = "confidence",
        relative_improvement: float = 0.0,
        ci_multiplier: float = 1.96,
        lower_quantile: float = 0.001,
        upper_quantile: float = 1.0,
        plot_ratio=True,
        save_path=None,
        ratio_name="Optimization Ratio",
        threshold_transform=lambda x: x, W=W, height_ratio=1 / 3.1
):
    if improvement_metric != "confidence":
        raise ValueError("Only improvement_metric='confidence' is supported.")

    raw_conf_t = results["all_conf_transformed"]
    raw_conf_o = results["all_conf"]

    # Calculate quantiles in original space
    ql_raw = np.quantile(raw_conf_t, lower_quantile)
    qh_raw = np.quantile(raw_conf_t, upper_quantile)

    # Check if the transform flips the order (monotonically decreasing). Transforms can be used to swap between in and out of distribution scores.
    ql_trans = threshold_transform(ql_raw)
    qh_trans = threshold_transform(qh_raw)
    is_decreasing = ql_trans > qh_trans

    all_conf = threshold_transform(results["all_conf"])
    all_conf_t = threshold_transform(results["all_conf_transformed"])
    opt_conf = threshold_transform(results["optimized_all_conf"])
    opt_conf_t = threshold_transform(results["optimized_all_conf_transformed"])

    corr = results["correct_predicted"]
    corr_t = results["correct_predicted_transformed"]
    opt_corr = results["optimized_correct_predicted"]
    opt_corr_t = results["optimized_correct_predicted_transformed"]

    # Transform range for relative improvement scaling
    min_conf_trans = float(min(all_conf.min(), all_conf_t.min()))
    max_conf_trans = float(max(all_conf.max(), all_conf_t.max()))
    conf_range_trans = abs(max_conf_trans - min_conf_trans)

    # We iterate through the ORIGINAL scale to ensure we pick the same
    # samples as the original graph, then map the threshold for the x-axis
    raw_threshold_steps = np.linspace(ql_raw, qh_raw, 400)
    plot_thresholds = threshold_transform(raw_threshold_steps)

    n_runs = opt_conf.shape[1]
    acc_orig_runs, acc_trans_runs = [], []
    ratio_orig_runs, ratio_trans_runs = [], []

    for r in range(n_runs):
        acc_o, acc_t = [], []
        ratio_o, ratio_t = [], []

        for i in range(len(raw_threshold_steps)):
            raw_thr = raw_threshold_steps[i]
            trans_thr = plot_thresholds[i]
            # if score is decreasing swap comparisons for threshold.
            if is_decreasing:
                applied_o = all_conf >= trans_thr
                applied_t = all_conf_t >= trans_thr
            else:
                applied_o = all_conf <= trans_thr
                applied_t = all_conf_t <= trans_thr

            improved_o = opt_conf[:, r] >= all_conf if not is_decreasing else opt_conf[:, r] <= all_conf
            improved_t = opt_conf_t[:, r] >= all_conf_t if not is_decreasing else opt_conf_t[:, r] <= all_conf_t

            if relative_improvement > 0.0:
                rel_imp_o = abs(opt_conf[:, r] - all_conf) / conf_range_trans
                rel_imp_t = abs(opt_conf_t[:, r] - all_conf_t) / conf_range_trans
                improved_o = np.logical_and(improved_o, rel_imp_o >= relative_improvement)
                improved_t = np.logical_and(improved_t, rel_imp_t >= relative_improvement)

            attempted_o = np.logical_and(applied_o, improved_o)
            attempted_t = np.logical_and(applied_t, improved_t)

            ratio_o.append(applied_o.mean())
            ratio_t.append(applied_t.mean())

            use_opt_o = attempted_o if use_only_improved else applied_o
            use_opt_t = attempted_t if use_only_improved else applied_t

            final_corr_o = np.where(use_opt_o, opt_corr[:, r], corr)
            final_corr_t = np.where(use_opt_t, opt_corr_t[:, r], corr_t)

            acc_o.append(final_corr_o.mean())
            acc_t.append(final_corr_t.mean())

        acc_orig_runs.append(acc_o)
        acc_trans_runs.append(acc_t)
        ratio_orig_runs.append(ratio_o)
        ratio_trans_runs.append(ratio_t)

    # Convert to numpy for CI calculation
    acc_orig_runs, acc_trans_runs = np.array(acc_orig_runs), np.array(acc_trans_runs)
    ratio_orig_runs, ratio_trans_runs = np.array(ratio_orig_runs), np.array(ratio_trans_runs)

    def compute_ci(y):
        n = y.shape[0]
        mean = y.mean(axis=0)
        se = y.std(axis=0, ddof=1) / np.sqrt(n) if n > 1 else np.zeros_like(mean)
        return mean, mean - ci_multiplier * se, mean + ci_multiplier * se

    fig, ax1 = plt.subplots(figsize=(W, W * height_ratio))
    tab10 = plt.get_cmap("tab10")
    col_o, col_t, col_ro, col_rt = tab10(0), tab10(1), tab10(2), tab10(3)

    ax1.grid(True, linestyle="--", alpha=0.6)

    m_o, lo_o, hi_o = compute_ci(acc_orig_runs)
    m_t, lo_t, hi_t = compute_ci(acc_trans_runs)

    ax1.plot(plot_thresholds, m_o, label="Acc. Original", color=col_o)
    ax1.fill_between(plot_thresholds, lo_o, hi_o, alpha=0.25, color=col_o)
    ax1.plot(plot_thresholds, m_t, label="Acc. Transformed", color=col_t)
    ax1.fill_between(plot_thresholds, lo_t, hi_t, alpha=0.25, color=col_t)

    ax1.set_xlabel("Threshold", fontsize=9)
    ax1.set_ylabel("Accuracy", fontsize=9)

    if plot_ratio:
        ax2 = ax1.twinx()
        rm_o, rlo_o, rhi_o = compute_ci(ratio_orig_runs)
        rm_t, rlo_t, rhi_t = compute_ci(ratio_trans_runs)
        ax2.plot(plot_thresholds, rm_o, linestyle="--", color=col_ro, label="Ratio Original")
        ax2.plot(plot_thresholds, rm_t, linestyle="--", color=col_rt, label="Ratio Transformed")
        ax2.set_ylabel(ratio_name)

    ax1.set_xlim([ql_trans, qh_trans])
    if plot_ratio:
        ax2.set_xlim([ql_trans, qh_trans])

    col_hline = tab10(4)
    if "augmnet_acc_on_in_dist" in globals():
        ax1.axhline(
            y=augmnet_acc_on_in_dist,
            color=col_hline,
            linestyle='--',
            label='Acc. Baseline',
        )

    #Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    if plot_ratio:
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labels1 + labels2, loc="center left", fontsize=8, framealpha=0.5,
                   labelspacing=0.2, )
    else:
        ax1.legend(lines1, labels1, loc="center left", fontsize=8, framealpha=0.5, labelspacing=0.2, )

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight")

    plt.show()

    print(f"[{dataset_name}] Direction: {'Decreasing' if is_decreasing else 'Increasing'}")
    print(f"Range: {ql_trans:.4f} to {qh_trans:.4f}")


In [ ]:
figure_path = os.path.join(current_path, "experiment_files", "export", "fig2", "comp_tradeoff", dataset, transform_name)
os.makedirs(figure_path, exist_ok=True)

In [ ]:

from search.random_search import RSLR

optimizer = RSLR(initial_samples=46, local_runs=2, local_max_steps=3, local_opt_kwargs={"lr": 0.1})

if dataset == "tu_berlin":
    optimizer = RSLR(initial_samples=60, local_runs=1, local_max_steps=0, local_opt_kwargs={"lr": 0.1})

In [ ]:

import torch


In [ ]:
os.makedirs(os.path.dirname(cache_path(f"cached_confidences.npz")), exist_ok=True)

In [ ]:
test_loader_determinism(test_loader)
test_loader_determinism(test_loader_transformed)
#both must return the exact same data at the exact same position.

In [ ]:
results = calculate_confidences_and_cache(
    optimizer=optimizer,
    best_problem=best_problem,
    conf_fn=conf,
    test_loader=test_loader,
    test_loader_transformed=test_loader_transformed,
    device=device,
    cache_path=cache_path(f"cached_confidences_{best_detector}.npz"),
    n_runs=5,
)
plot_confidence_threshold_results(results, dataset_name=dataset,
                                  save_path=os.path.join(figure_path, "confidence_threshold.pdf"),
                                  threshold_transform=lambda y: 1 / y - 1)


In [ ]:
plot_confidence_threshold_results(results, dataset_name=dataset,
                                  save_path=os.path.join(figure_path, "confidence_threshold_small.pdf"), W=W * 2 / 3,
                                  height_ratio=1 / 3.1 * 3 / 2, threshold_transform=lambda y: 1 / y - 1)


In [ ]:
plot_confidence_threshold_results(results, dataset_name=dataset, relative_improvement=0.1,
                                  save_path=os.path.join(figure_path, "confidence_threshold_rel_imp_10.pdf"),
                                  threshold_transform=lambda y: 1 / y - 1)

In [ ]:
results

In [ ]:
from src.utils.augments import build_default_augmentations, small_affine_augment_2d

In [ ]:
if is_image_data:
    results = calculate_confidences_and_cache(
        optimizer=optimizer,
        best_problem=best_problem,
        conf_fn=conf,
        test_loader=test_loader,
        test_loader_transformed=test_loader_transformed,
        device=device,
        n_runs=5, cache_path=cache_path(f"cached_confidences_blur_aug.npz"),
        augment_true_func=small_affine_augment_2d,
    )
    plot_confidence_threshold_results(results, dataset_name=dataset, use_only_improved=False)
    plot_confidence_threshold_results(results, dataset_name=dataset,
                                      save_path=os.path.join(figure_path, "confidence_threshold_blur.pdf"))


In [ ]:
model_dir_path

In [ ]:
from search.random_search import RSLR

optimizer_120 = RSLR(initial_samples=120 - 27, local_runs=3, local_max_steps=4, local_opt_kwargs={"lr": 0.1})
if dataset == "tu_berlin":
    optimizer_120 = RSLR(initial_samples=120, local_runs=1, local_max_steps=0, local_opt_kwargs={"lr": 0.1})

from src.utils.augments import small_affine_augment_2d, ComposeAugmentations, random_blur_or_sharpen
from src.utils.sampling_strategy import TransformLatentSamplingStrategy
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
from src.utils.transformation_problem import TransformationProblem

architecture_half = architecture + "_half"

if is_image_data:
    transform_true_function = small_affine_augment_2d
    affine_augment = ComposeAugmentations([
        lambda x: random_blur_or_sharpen(x, p=0.8, prob_blur=0.5,
                                         blur_ks_choices=(3, 5), blur_sigma_range=(0.2, 1.8),
                                         usm_ksize=5, usm_sigma_range=(0.5, 1.5),
                                         usm_amount_range=(0.5, 1.3), clamp=True),

    ])
else:
    transform_true_function = None
    affine_augment = None


def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y


energy_model2_half = get_network(dataset_info, architecture_half, num_classes=1).to(device)
negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    =transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)
energy_conf_half = train_or_load_energy_model(
    energy_model2_half, model_dir_path, f"{modelname}_energy2_half", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs // 2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)

energy_conf_half.to(device).eval()
problem_energy_half = TransformationProblem(energy_conf_half, transform_seq, consolidate_method="consolidate_simple",
                                            max_batch_size=dataset_info.batch_size_search)


def savepath_later_layer(label: str) -> str:
    model_dir_path = os.path.join(current_path, "experiment_files", "models")
    embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
    # Add results dir and helper for save paths
    results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,
                                    "comparison_supervised_methods")
    os.makedirs(results_dir_path, exist_ok=True)

    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, transform_name, f"{safe}.json")


res_120 = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_120, problem=problem_energy_half,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath_later_layer("learned_energy_confidence_half_transformed_120"), show_progress=True,
    repeats=5)

res_120_real_data = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_120, problem=problem_energy_half,
    test_loader=test_loader, max_batch_override=dataset_info.batch_size_search, show_progress=True, save_path=
    savepath_later_layer("learned_energy_confidence_half_transformed_120_untransformed"),
    repeats=5)

del energy_model2_half
del energy_conf_half

print(res_120["accuracy_mean"])

In [ ]:
gc.collect()

In [ ]:
test_loader_transformed_quarter = torch.utils.data.DataLoader(
    test_loader_transformed.dataset,
    batch_size=max(dataset_info.batch_size // 4, 1),
    shuffle=False,
    num_workers=test_loader_transformed.num_workers,
    pin_memory=test_loader_transformed.pin_memory,
    persistent_workers=test_loader_transformed.persistent_workers
)
test_loader_quarter = torch.utils.data.DataLoader(
    test_loader.dataset,
    batch_size=max(dataset_info.batch_size // 4, 1),
    shuffle=False,
    num_workers=test_loader.num_workers,
    pin_memory=test_loader.pin_memory,
    persistent_workers=test_loader.persistent_workers
)

calculate_confidences_and_cache(
    optimizer=optimizer_120,
    best_problem=problem_energy_half,
    conf_fn=problem_energy_half.confidence_module,
    test_loader=test_loader_quarter,
    test_loader_transformed=test_loader_transformed_quarter,
    device=device,
    n_runs=5, cache_path=cache_path(f"cached_confidences_energy.npz"),
    augment_true_func=None,
)

#plot the results
results_energy = np.load(cache_path(f"cached_confidences_energy.npz"), allow_pickle=True)
#compare the two
plot_confidence_threshold_results(results_energy, dataset_name=dataset,
                                  save_path=os.path.join(figure_path, "confidence_threshold_energy_vs_base.pdf"),
                                  upper_quantile=0.99, threshold_transform=lambda y: -y)



In [ ]:
plot_confidence_threshold_results(results_energy, dataset_name=dataset, relative_improvement=0.1,
                                  save_path=os.path.join(figure_path,
                                                         "confidence_threshold_energy_vs_base_rel_imp_10.pdf"),
                                  upper_quantile=0.99)


In [ ]:
if is_image_data:
    calculate_confidences_and_cache(
        optimizer=optimizer_120,
        best_problem=problem_energy_half,
        conf_fn=problem_energy_half.confidence_module,
        test_loader=test_loader_quarter,
        test_loader_transformed=test_loader_transformed_quarter,
        device=device,
        n_runs=5, cache_path=cache_path(f"cached_confidences_energy_blur_aug.npz"),
        augment_true_func=small_affine_augment_2d,
    )

    #plot the results
    results_energy = np.load(cache_path(f"cached_confidences_energy_blur_aug.npz"), allow_pickle=True)
    #compare the two
    plot_confidence_threshold_results(results_energy, dataset_name=dataset, save_path=os.path.join(figure_path,
                                                                                                   "confidence_threshold_energy_vs_base_blur.pdf"),
                                      threshold_transform=lambda y: -y)



In [ ]:

from src.utils.augments import small_affine_augment_2d, ComposeAugmentations, random_blur_or_sharpen
from src.utils.sampling_strategy import TransformLatentSamplingStrategy
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
from src.utils.transformation_problem import TransformationProblem

architecture_quarter = architecture + "_quarter"

if is_image_data:
    transform_true_function = small_affine_augment_2d
    affine_augment = ComposeAugmentations([
        lambda x: random_blur_or_sharpen(x, p=0.8, prob_blur=0.5,
                                         blur_ks_choices=(3, 5), blur_sigma_range=(0.2, 1.8),
                                         usm_ksize=5, usm_sigma_range=(0.5, 1.5),
                                         usm_amount_range=(0.5, 1.3), clamp=True),

    ])
else:
    transform_true_function = None
    affine_augment = None


def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y


from search.random_search import RSLR

optimizer_240 = RSLR(initial_samples=195, local_runs=5, local_max_steps=4, local_opt_kwargs={"lr": 0.1})
if dataset == "tu_berlin":
    optimizer_240 = RSLR(initial_samples=240, local_runs=1, local_max_steps=0, local_opt_kwargs={"lr": 0.1})
if dataset == "modelnet10":
    optimizer_240 = RSLR(initial_samples=180 - 27, local_runs=3, local_max_steps=4, local_opt_kwargs={"lr": 0.1})

optimizer_240_include_init = RSLR(initial_samples=195, local_runs=5, local_max_steps=4, local_opt_kwargs={"lr": 0.1},
                                  include_zero_always=True)
if dataset == "tu_berlin":
    optimizer_240_include_init = RSLR(initial_samples=240, local_runs=1, local_max_steps=0,
                                      local_opt_kwargs={"lr": 0.1}, include_zero_always=True)
if dataset == "modelnet10":
    optimizer_240_include_init = RSLR(initial_samples=180 - 27, local_runs=3, local_max_steps=4,
                                      local_opt_kwargs={"lr": 0.1}, include_zero_always=True)

energy_model2_quarter = get_network(dataset_info, architecture_quarter, num_classes=1).to(device)
negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    =transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)
energy_conf_quarter = train_or_load_energy_model(
    energy_model2_quarter, model_dir_path, f"{modelname}_energy2_quarter", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs // 2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)

energy_conf_quarter.to(device).eval()
problem_energy_quarter = TransformationProblem(energy_conf_quarter, transform_seq,
                                               consolidate_method="consolidate_simple")


def savepath_later_layer(label: str) -> str:
    model_dir_path = os.path.join(current_path, "experiment_files", "models")
    embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
    # Add results dir and helper for save paths
    results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,
                                    "comparison_supervised_methods")
    os.makedirs(results_dir_path, exist_ok=True)

    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, transform_name, f"{safe}.json")


name = "learned_energy_confidence_quarter_transformed_240"
name_un = "learned_energy_confidence_quarter_untransformed_240"
name_un_included = "learned_energy_confidence_quarter_untransformed_240_inc"
if dataset == "modelnet10":
    name = "learned_energy_confidence_quarter_transformed_180"
    name_un = "learned_energy_confidence_quarter_untransformed_180"
    name_un_included = "learned_energy_confidence_quarter_untransformed_180_inc"

res_large_transformed = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_240, problem=problem_energy_quarter,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath_later_layer(name), show_progress=True,
    repeats=5)

res_large_real_data = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_240, problem=problem_energy_quarter,
    test_loader=test_loader, max_batch_override=dataset_info.batch_size_search, show_progress=True, save_path=
    savepath_later_layer(name_un),
    repeats=5)

res_large_real_data_included = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_240_include_init, problem=problem_energy_quarter,
    test_loader=test_loader, max_batch_override=dataset_info.batch_size_search, show_progress=True, save_path=
    savepath_later_layer(name_un_included),
    repeats=5)

print(res_large_real_data)
print(res_large_real_data_included)

del energy_model2_quarter
del energy_conf_quarter








In [ ]:
calculate_confidences_and_cache(
    optimizer=optimizer_240,
    best_problem=problem_energy_quarter,
    conf_fn=problem_energy_quarter.confidence_module,
    test_loader=test_loader_quarter,
    test_loader_transformed=test_loader_transformed_quarter,
    device=device,
    n_runs=5, cache_path=cache_path(f"cached_confidences_energy_quarter.npz"),
    augment_true_func=None,
)

#plot the results
results_energy_quarter = np.load(cache_path(f"cached_confidences_energy_quarter.npz"), allow_pickle=True)
#compare the two
plot_confidence_threshold_results(results_energy_quarter, dataset_name=dataset, save_path=os.path.join(figure_path,
                                                                                                       "confidence_threshold_energy_quarter_vs_base.pdf"),
                                  upper_quantile=0.99, threshold_transform=lambda y: -y)

In [ ]:
plot_confidence_threshold_results(results_energy_quarter, dataset_name=dataset, relative_improvement=0.1,
                                  save_path=os.path.join(figure_path,
                                                         "confidence_threshold_energy_quarter_vs_base_rel_imp_10.pdf"),
                                  upper_quantile=0.99, threshold_transform=lambda y: -y)

In [ ]:
if is_image_data:
    calculate_confidences_and_cache(
        optimizer=optimizer_240,
        best_problem=problem_energy_quarter,
        conf_fn=problem_energy_quarter.confidence_module,
        test_loader=test_loader_quarter,
        test_loader_transformed=test_loader_transformed_quarter,
        device=device,
        n_runs=5, cache_path=cache_path(f"cached_confidences_energy_quarter_blur_aug.npz"),
        augment_true_func=small_affine_augment_2d,
    )

    #plot the results
    results_energy_quarter = np.load(cache_path(f"cached_confidences_energy_quarter_blur_aug.npz"), allow_pickle=True)
    #compare the two
    plot_confidence_threshold_results(results_energy_quarter, dataset_name=dataset, save_path=os.path.join(figure_path,
                                                                                                           "confidence_threshold_energy_quarter_vs_base_blur.pdf"),
                                      threshold_transform=lambda y: -y)

In [ ]:
@torch.no_grad()
def calculate_confidences_and_cache_augmented(
        model_augmented,
        conf_fn,
        test_loader,
        test_loader_transformed,
        device,
        cache_path: str = "cached_confidences.npz",
        force_recompute: bool = False,
):
    """
    Computes and caches:
      - base model: confidence + correctness
      - augmented model: logits + correctness (NO augmented confidence)
    Shapes are kept compatible with optimization-based plotting code.
    """

    # Load cached results
    if os.path.exists(cache_path) and not force_recompute:
        print(f"Loading cached results from {cache_path}")
        return dict(np.load(cache_path, allow_pickle=True))

    print("=== Calculating base confidences ===")

    def eval_base(loader):
        all_conf, correct = [], []

        for x_test, y_test in tqdm.tqdm(loader):
            x_test = x_test.to(device)
            conf_values, logits = conf_fn(x_test)

            pred_y = torch.argmax(logits, dim=1).cpu()
            correct.append((pred_y == y_test).numpy())
            all_conf.append(conf_values.cpu().numpy())

        return np.concatenate(correct), np.concatenate(all_conf)

    def eval_augmented(loader):
        correct = []

        for x_test, y_test in tqdm.tqdm(loader):
            x_test = x_test.to(device)

            logits = model_augmented(x_test)
            pred_y = torch.argmax(logits, dim=1).cpu()

            correct.append((pred_y == y_test).numpy())

        return np.concatenate(correct)

    # Base classifier
    base_correct, base_conf = eval_base(test_loader)
    base_correct_t, base_conf_t = eval_base(test_loader_transformed)

    print("\n=== Calculating augmented model predictions ===")

    aug_correct = eval_augmented(test_loader)
    aug_correct_t = eval_augmented(test_loader_transformed)

    # Make shapes consistent with optimizer output
    aug_correct = aug_correct[:, None]  # (N,1)
    aug_correct_t = aug_correct_t[:, None]  # (N,1)
    base_conf = base_conf[:, None]  # (N,1)
    base_conf_t = base_conf_t[:, None]  # (N,1)

    results = {
        # base model
        "correct_predicted": base_correct,
        "all_conf": base_conf.squeeze(),  # keep original base shape for baseline
        "correct_predicted_transformed": base_correct_t,
        "all_conf_transformed": base_conf_t.squeeze(),

        # augmented model (formatted like optimized runs)
        "optimized_correct_predicted": aug_correct,
        "optimized_correct_predicted_transformed": aug_correct_t,

        # confidence is identical to base, but plotter expects these keys with shape (N,1)
        "optimized_all_conf": base_conf,
        "optimized_all_conf_transformed": base_conf_t,
    }

    np.savez_compressed(cache_path, **results)

    return results


model_augmented = get_network(dataset_info, architecture, num_classes=n_classes).to(device)
modelname_augmented = f"{dataset}_{architecture}_augmented"

train_and_get_model(model_augmented, model_dir_path, modelname_augmented, None, val_loader, trainer_kwargs={
    "accelerator": "auto",
    "max_epochs": dataset_info.epochs,
    "precision": "16-mixed",
}, load_if_exists=True)

cache_path_augmented = cache_path(f"cached_confidences_augmented_model.npz")

results_augmented = calculate_confidences_and_cache_augmented(
    model_augmented=model_augmented,
    conf_fn=best_problem.confidence_module,
    test_loader=test_loader,
    test_loader_transformed=test_loader_transformed,
    device=device,
    cache_path=cache_path_augmented,
)

plot_confidence_threshold_results(results_augmented, dataset_name=dataset, plot_ratio=True,
                                  save_path=os.path.join(figure_path, "confidence_threshold_augmented.pdf"),
                                  threshold_transform=lambda y: 1 / y - 1)



In [ ]:
@torch.no_grad()
def calculate_confidences_and_cache_augmented(
        model_augmented,
        conf_fn,
        test_loader,
        test_loader_transformed,
        device,
        cache_path: str = "cached_confidences.npz",
        force_recompute: bool = False,
):
    """
    Computes and caches:
      - base model: confidence + correctness
      - augmented model: logits + correctness (NO augmented confidence)
    Shapes are kept compatible with optimization-based plotting code.
    """

    # Load cached results
    if os.path.exists(cache_path) and not force_recompute:
        print(f"Loading cached results from {cache_path}")
        return dict(np.load(cache_path, allow_pickle=True))

    print("=== Calculating base confidences ===")

    def eval_base(loader):
        all_conf, correct = [], []

        for x_test, y_test in tqdm.tqdm(loader):
            x_test = x_test.to(device)
            conf_values, logits = conf_fn(x_test)
            if logits is None:
                logits = model(x_test)

            pred_y = torch.argmax(logits, dim=1).cpu()
            correct.append((pred_y == y_test).numpy())
            all_conf.append(conf_values.cpu().numpy())

        return np.concatenate(correct), np.concatenate(all_conf)

    def eval_augmented(loader):
        correct = []

        for x_test, y_test in tqdm.tqdm(loader):
            x_test = x_test.to(device)

            logits = model_augmented(x_test)
            pred_y = torch.argmax(logits, dim=1).cpu()

            correct.append((pred_y == y_test).numpy())

        return np.concatenate(correct)

    # Base classifier
    base_correct, base_conf = eval_base(test_loader)
    base_correct_t, base_conf_t = eval_base(test_loader_transformed)

    print("\n=== Calculating augmented model predictions ===")

    aug_correct = eval_augmented(test_loader)
    aug_correct_t = eval_augmented(test_loader_transformed)

    # Make shapes consistent with optimizer output
    aug_correct = aug_correct[:, None]  # (N,1)
    aug_correct_t = aug_correct_t[:, None]  # (N,1)
    base_conf = base_conf[:, None]  # (N,1)
    base_conf_t = base_conf_t[:, None]  # (N,1)

    results = {
        # base model
        "correct_predicted": base_correct,
        "all_conf": base_conf.squeeze(),
        "correct_predicted_transformed": base_correct_t,
        "all_conf_transformed": base_conf_t.squeeze(),

        # augmented model
        "optimized_correct_predicted": aug_correct,
        "optimized_correct_predicted_transformed": aug_correct_t,

        "optimized_all_conf": base_conf,
        "optimized_all_conf_transformed": base_conf_t,
    }

    np.savez_compressed(cache_path, **results)

    return results


model_augmented = get_network(dataset_info, architecture, num_classes=n_classes).to(device)
modelname_augmented = f"{dataset}_{architecture}_augmented"

train_and_get_model(model_augmented, model_dir_path, modelname_augmented, None, val_loader, trainer_kwargs={
    "accelerator": "auto",
    "max_epochs": dataset_info.epochs,
    "precision": "16-mixed",
}, load_if_exists=True)

cache_path_augmented = cache_path(f"cached_confidences_augmented_model_learned.npz")

results_augmented = calculate_confidences_and_cache_augmented(
    model_augmented=model_augmented,
    conf_fn=problem_energy_half.confidence_module,
    test_loader=test_loader,
    test_loader_transformed=test_loader_transformed,
    device=device,
    cache_path=cache_path_augmented,
)

plot_confidence_threshold_results(results_augmented, dataset_name=dataset, plot_ratio=True,
                                  save_path=os.path.join(figure_path, "confidence_threshold_augmented_learned.pdf"),
                                  ratio_name="Ratio", threshold_transform=lambda y: -y)



In [ ]:
def create_mixed_arrays(normal_arr, trans_arr, ratio: float, seed: int = 42):
    """
    Mixes elements from normal and transformed arrays based on a given ratio.
    Keeps the size of the returned array equal to the original array size.
    """
    np.random.seed(seed)
    n_samples = len(normal_arr)
    n_trans = int(n_samples * ratio)

    # Randomly select indices to replace with transformed versions
    indices = np.random.choice(n_samples, n_trans, replace=False)

    # Create a copy of the normal array as the base
    mixed_arr = np.copy(normal_arr)
    mixed_arr[indices] = trans_arr[indices]

    return mixed_arr


def evaluate_mixed_scenarios(
        val_results: dict,
        test_results: dict,
        ratios=[0.1, 0.25, 0.5, 0.75, 0.9],
        lower_quantile: float = 0.001,
        upper_quantile: float = 1.0,
        threshold_transform=lambda x: x,
):
    # Determine directionality from val set
    raw_conf_t = val_results["all_conf_transformed"]
    ql_raw = np.quantile(raw_conf_t, lower_quantile)
    qh_raw = np.quantile(raw_conf_t, upper_quantile)
    is_decreasing = threshold_transform(ql_raw) > threshold_transform(qh_raw)

    # Define search space for thresholds using the validation bounds
    raw_threshold_steps = np.linspace(ql_raw, qh_raw, 400)
    plot_thresholds = threshold_transform(raw_threshold_steps)

    n_runs = val_results["optimized_all_conf"].shape[1]

    print(f"Evaluation Mode | Direction: {'Decreasing' if is_decreasing else 'Increasing'}")
    print(f"Ratios evaluated: {ratios}\n" + "=" * 50)

    for ratio in ratios:
        # --- 1. MIX BASE VALIDATION & TEST DATA (Fixed environment splits) ---
        val_seed = 42
        test_seed = 100

        val_conf = create_mixed_arrays(val_results["all_conf"], val_results["all_conf_transformed"], ratio,
                                       seed=val_seed)
        val_corr = create_mixed_arrays(val_results["correct_predicted"], val_results["correct_predicted_transformed"],
                                       ratio, seed=val_seed)

        test_conf = create_mixed_arrays(test_results["all_conf"], test_results["all_conf_transformed"], ratio,
                                        seed=test_seed)
        test_corr = create_mixed_arrays(test_results["correct_predicted"],
                                        test_results["correct_predicted_transformed"], ratio, seed=test_seed)

        v_conf_t = threshold_transform(val_conf)
        t_conf_t = threshold_transform(test_conf)

        # Pre-mix and transform optimization runs for validation to align indices
        val_opt_confs_mixed = []
        val_opt_corrs_mixed = []
        for r in range(n_runs):
            v_opt_c = create_mixed_arrays(val_results["optimized_all_conf"][:, r],
                                          val_results["optimized_all_conf_transformed"][:, r], ratio, seed=val_seed)
            v_opt_r = create_mixed_arrays(val_results["optimized_correct_predicted"][:, r],
                                          val_results["optimized_correct_predicted_transformed"][:, r], ratio,
                                          seed=val_seed)
            val_opt_confs_mixed.append(threshold_transform(v_opt_c))
            val_opt_corrs_mixed.append(v_opt_r)

        # Look for the single threshold that maximizes the mean validation accuracy across all runs
        best_mean_val_acc = -1.0
        chosen_threshold = None

        for idx, raw_thr in enumerate(raw_threshold_steps):
            trans_thr = plot_thresholds[idx]
            val_accs_for_thr = []

            for r in range(n_runs):
                v_opt_conf_t = val_opt_confs_mixed[r]
                v_opt_corr = val_opt_corrs_mixed[r]

                if is_decreasing:
                    applied = v_conf_t >= trans_thr
                    improved = v_opt_conf_t <= v_conf_t
                else:
                    applied = v_conf_t <= trans_thr
                    improved = v_opt_conf_t >= v_conf_t

                attempted = np.logical_and(applied, improved)
                final_corr_val = np.where(attempted, v_opt_corr, val_corr)
                val_accs_for_thr.append(final_corr_val.mean())

            mean_val_acc = np.mean(val_accs_for_thr)
            if mean_val_acc > best_mean_val_acc:
                best_mean_val_acc = mean_val_acc
                chosen_threshold = trans_thr

        # Evaluate on threshold using found threshold.
        test_accuracies_per_run = []
        baseline_test_accuracies = []

        for r in range(n_runs):
            t_opt_c = create_mixed_arrays(test_results["optimized_all_conf"][:, r],
                                          test_results["optimized_all_conf_transformed"][:, r], ratio, seed=test_seed)
            t_opt_r = create_mixed_arrays(test_results["optimized_correct_predicted"][:, r],
                                          test_results["optimized_correct_predicted_transformed"][:, r], ratio,
                                          seed=test_seed)
            t_opt_conf_t = threshold_transform(t_opt_c)

            if is_decreasing:
                test_applied = t_conf_t >= chosen_threshold
                test_improved = t_opt_conf_t <= t_conf_t
            else:
                test_applied = t_conf_t <= chosen_threshold
                test_improved = t_opt_conf_t >= t_conf_t

            test_attempted = np.logical_and(test_applied, test_improved)
            final_corr_test = np.where(test_attempted, t_opt_r, test_corr)

            test_accuracies_per_run.append(final_corr_test.mean())
            baseline_test_accuracies.append(test_corr.mean())

        # --- 4. DISPLAY RESULTS ---
        print(f"Ratio (Transformed/Total): {ratio:.2f}")
        print(f"  -> Baseline Test Acc (No Opt) : {np.mean(baseline_test_accuracies):.4f}")
        print(
            f"  -> Optimized Test Acc          : {np.mean(test_accuracies_per_run):.4f} ± {np.std(test_accuracies_per_run):.4f}")
        print(
            f"  -> Selected Single Threshold   : {chosen_threshold:.4f} (Validation Mean Acc: {best_mean_val_acc:.4f})")
        print("-" * 50)


# 1. Compute and cache Validation parameters
val_results = calculate_confidences_and_cache(
    optimizer=optimizer,
    best_problem=best_problem,
    conf_fn=best_problem.confidence_module,
    test_loader=val_loader,
    test_loader_transformed=val_loader_transformed,
    device=device,
    n_runs=5,
    cache_path=cache_path("cached_val_confidences.npz"),
)
# 2. Compute and cache Test parameters
test_results = calculate_confidences_and_cache(
    optimizer=optimizer,
    best_problem=best_problem,
    conf_fn=best_problem.confidence_module,
    test_loader=test_loader,
    test_loader_transformed=test_loader_transformed,
    device=device,
    n_runs=5,
    cache_path=cache_path("cached_test_confidences.npz"),
)

# 3. Process Mixtures and find generalized metrics
evaluate_mixed_scenarios(
    val_results=val_results,
    test_results=test_results,
    ratios=[0.1, 0.25, 0.5, 0.75, 0.9]
)

In [ ]:
#Test of methods. Thrshold has to be st based on values from the previos results. The choosen ones makes sense for bigger emnist.

The following is validation of specific thresholds. The first is a threshold test on both datasets. The later is a threshold test on a mixture.


In [ ]:

threshold = 0.75

import torch
import numpy as np
import tqdm


def eval_threshold_run_remainder(loader, label, threshold, device, conf, optimizer, best_problem):
    n_total = 0
    n_accept = 0
    correct_final = 0
    correct_base = 0

    with torch.no_grad():
        for x_batch, y_batch in tqdm.tqdm(loader, desc=label):
            # ensure tensors
            x_batch = x_batch.to(device)
            y_cpu = y_batch.cpu()  # keep CPU copy for comparisons

            # base confidences and predictions
            conf_vals, logits = conf(x_batch)
            conf_cpu = conf_vals.cpu().view(-1)  # shape (B,)

            base_pred = torch.argmax(logits, dim=1).cpu()
            base_correct = (base_pred == y_cpu)

            # decide which to accept (do NOT optimize) and which to optimize
            accept_mask = conf_cpu >= threshold
            accept_mask_torch = accept_mask.to(device)  # for indexing on device tensors if needed

            # prepare final_correct array (torch bool on CPU)
            final_correct_batch = base_correct.clone()

            # If there are items to optimize, run optimizer on the remainder only
            if (~accept_mask).any():
                # select subset to optimize
                idx_rem = torch.nonzero(~accept_mask, as_tuple=False).squeeze(1)
                x_rem = x_batch[idx_rem]  # already on device
                y_rem_cpu = y_cpu[idx_rem]  # cpu labels

                # run optimizer only on remainder
                p, error, _ = optimizer.optimize(best_problem, x_rem)

                # apply transforms produced by optimizer and recompute confidence/logits
                x_opt = best_problem.transform_sequence.transform(x_rem, p)
                conf_opt_vals, logits_opt = conf(x_opt)
                opt_pred = torch.argmax(logits_opt, dim=1).cpu()
                opt_correct = (opt_pred == y_rem_cpu)

                # place optimized correctness back into final_correct_batch
                final_correct_batch[idx_rem] = opt_correct

            # update accept counters and base counters
            correct_base += int(base_correct.sum().item())
            n_accept += int(accept_mask.sum().item())
            batch_n = final_correct_batch.numel()
            n_total += batch_n
            correct_final += int(final_correct_batch.sum().item())

    acc_final = correct_final / n_total if n_total else float("nan")
    acc_base = correct_base / n_total if n_total else float("nan")

    print(f"{label}: final acc={acc_final:.4f} (accepted={n_accept}, total={n_total})  | base acc={acc_base:.4f}")


print()
threshold = 0.7
eval_threshold_run_remainder(test_loader, "Original test", threshold, device, conf, optimizer, best_problem)
eval_threshold_run_remainder(test_loader_transformed, "Transformed test", threshold, device, conf, optimizer,
                             best_problem)

In [ ]:
conf

In [ ]:
import numpy as np
import torch
import tqdm


# =====================================================================
# 1. LIVE INFERENCE COMPONENTS
# =====================================================================

class MixedDataset(torch.utils.data.Dataset):
    def __init__(self, normal_ds, trans_ds, ratio: float, seed: int = 42):
        self.normal_ds = normal_ds
        self.trans_ds = trans_ds
        self.length = len(normal_ds)

        # Determine deterministic indices to corrupt/transform
        np.random.seed(seed)
        n_trans = int(self.length * ratio)
        self.trans_indices = set(np.random.choice(self.length, n_trans, replace=False))

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if idx in self.trans_indices:
            return self.trans_ds[idx]
        return self.normal_ds[idx]


@torch.no_grad()
def simulate_inference_run(
        optimizer,
        best_problem,
        conf_fn,
        model,
        inference_loader,
        threshold: float,
        is_decreasing: bool,
        device,
        threshold_transform=lambda x: x
) -> float:
    """
    Simulates production inference.
    Applies optimization ONLY to samples failing the confidence threshold.
    """
    correct_predictions = []
    total_optimized_count = 0
    total_samples = 0

    for x_batch, y_batch in tqdm.tqdm(inference_loader, desc="Simulating Production Inference"):
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        batch_size = x_batch.size(0)
        total_samples += batch_size

        # --- Phase 1: Quick Base Forward Pass ---
        conf_values, logits = conf_fn(x_batch)
        if logits is None:
            logits = model(x_batch)

        # Determine predictions without optimization
        base_preds = torch.argmax(logits, dim=1)

        # Apply transformation if necessary for thresholding scale
        transformed_conf = threshold_transform(conf_values)

        # --- Phase 2: Filter inputs based on the Threshold ---
        if is_decreasing:
            # Under-performing means confidence score is too high (e.g., OOD error scores)
            needs_optimization = transformed_conf >= threshold
        else:
            # Under-performing means confidence score is too low (standard probability metrics)
            needs_optimization = transformed_conf <= threshold

        # Collect indices needing optimization
        opt_indices = torch.where(needs_optimization)[0]
        total_optimized_count += len(opt_indices)

        # Container for final predictions of this batch
        final_preds = base_preds.clone()

        # --- Phase 3: Selective Optimization (Only run on worst subsets) ---
        if len(opt_indices) > 0:
            x_to_optimize = x_batch[opt_indices]

            # Optimize only the troubled subset
            p, _, _ = optimizer.optimize(best_problem, x_to_optimize)
            x_optimized = best_problem.transform_sequence.transform(x_to_optimize, p)

            # Re-evaluate the fixed variations
            opt_conf, opt_logits = conf_fn(x_optimized)
            if opt_logits is None:
                opt_logits = model(x_optimized)

            optimized_preds = torch.argmax(opt_logits, dim=1)

            # Optional verification step: enforce improvement rule
            opt_conf_transformed = threshold_transform(opt_conf)
            if is_decreasing:
                improved = opt_conf_transformed <= transformed_conf[opt_indices]
            else:
                improved = opt_conf_transformed >= transformed_conf[opt_indices]

            # Keep optimized predictions only if they actually helped confidence
            final_opt_preds = torch.where(improved, optimized_preds, base_preds[opt_indices])

            # Inject optimized predictions back into main batch array
            final_preds[opt_indices] = final_opt_preds

        # Track total accuracy performance
        correct_predictions.append((final_preds == y_batch).cpu().numpy())

    final_accuracy = np.concatenate(correct_predictions).mean()
    optimization_rate = (total_optimized_count / total_samples) * 100

    print("\n" + "=" * 60)
    print("LIVE PRODUCTION INFERENCE EXECUTION COMPLETED")
    print(f"  -> Input Inference Threshold : {threshold:.4f}")
    print(f"  -> Input Dataset Ratio        : {inference_loader.dataset.length} samples at {target_ratio * 100}% mix")
    print(f"  -> Final Test Accuracy        : {final_accuracy:.4f}")
    print(
        f"  -> Optimizer Activation Rate  : {optimization_rate:.2f}% ({total_optimized_count}/{total_samples} samples)")
    print("=" * 60 + "\n")

    return final_accuracy


# =====================================================================
# 2. CONFIGURATION AND INFERENCE EXECUTION
# =====================================================================

# Change these parameters manually as desired
SPECIFIED_THRESHOLD = 0.8047
target_ratio = 0.5
is_decreasing = False  # True if higher values equal lower confidence (e.g., error scores)

# --- Build the Mixed PyTorch Test Dataset for Production Inference ---
mixed_test_dataset = MixedDataset(
    normal_ds=dataset_test,
    trans_ds=dataset_dict['test_loader_transformed'].dataset,
    ratio=target_ratio
)

inference_loader = torch.utils.data.DataLoader(
    mixed_test_dataset,
    batch_size=test_loader.batch_size,
    shuffle=False
)

# --- Execute Pure Live Simulation ---
print(
    f"\n>>> Launching Pure Single-Run Inference Simulation (Ratio={target_ratio}, Threshold={SPECIFIED_THRESHOLD})...")
inference_accuracy = simulate_inference_run(
    optimizer=optimizer,
    best_problem=best_problem,
    conf_fn=best_problem.confidence_module,
    model=model,
    inference_loader=inference_loader,
    threshold=SPECIFIED_THRESHOLD,
    is_decreasing=is_decreasing,
    device=device
)